# Give your ADK travel agent a memory with Vertex AI Memory Bank

| Section | What you do |
|---|---|
| 1. Setup | Install, sign in, load `.env` |
| 2. Memory topics | What to remember, as sentences |
| 3. Profile schema | What to remember, as JSON fields |
| 4. Create the Memory Bank | Register topics + schema, saves `AGENT_ENGINE_ID` |
| 5. Write a fact | `memories.create` and `memories.generate` |
| 6. Read memories back | `retrieve`, `list`, similarity search |
| 7. Learn from a conversation | Topics and profile filled from a chat |
| 8. Clean up | Delete test memories or the bank |

Run the notebook from the repo root so it can find `.env`.

## 1. Setup

Install the packages into the same environment as the Jupyter kernel.

In [37]:
%pip install -q "google-adk[mcp]>=1.18" "google-cloud-aiplatform[agent-engines]>=1.148.1,<2" ipykernel


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [39]:
import os
from pathlib import Path


def load_dotenv(path: Path = Path(".env")) -> None:
    """Same rules as memories._load_dotenv: values already in the environment win."""
    if not path.exists():
        print(f"No {path.resolve()} found, using the current environment")
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


load_dotenv()

PROJECT_ID = os.environ.get("PROJECT_ID", "")          # or set it here
LOCATION = os.environ.get("LOCATION", "") or "us-central1"
AGENT_ENGINE_ID = os.environ.get("AGENT_ENGINE_ID", "")  # empty until section 3

APP_NAME = "travel_agent"   # must match the ADK app name (the agent folder)
USER_ID = "user"            # the default user in `adk web`

assert PROJECT_ID, "Set PROJECT_ID in .env or in this cell"
print("project:", PROJECT_ID, "| location:", LOCATION, "| engine:", AGENT_ENGINE_ID or "(not created yet)")

project: data-beat-labs | location: us-central1 | engine: (not created yet)


### Sign in to Google Cloud

Memory Bank calls use **Application Default Credentials (ADC)**, not `gcloud auth login`.

The next cell opens a browser window. Sign in, allow access, then come back here. The cell finishes when the browser says you're signed in.

> If you later see `RefreshError: invalid_grant`, your ADC token has expired or was revoked. Run this cell again.

In [ ]:
!gcloud auth application-default login

Create the Vertex AI client. If a later API call fails with `invalid_grant`, run the login cell again.

In [40]:
import vertexai

client = vertexai.Client(project=PROJECT_ID, location=LOCATION)
types = vertexai.types

## 2. Memory topics

A memory topic tells Memory Bank what to remember from a conversation. Anything that fits no topic is dropped.

“I'm vegetarian” and “I like walking tours” go to `travel_preferences`. “Our hotel is in Shinjuku” goes to `trip_context`.

In [24]:
MemoryTopic = types.MemoryBankCustomizationConfigMemoryTopic
CustomMemoryTopic = types.MemoryBankCustomizationConfigMemoryTopicCustomMemoryTopic

custom_topics = [
    MemoryTopic(custom_memory_topic=CustomMemoryTopic(
        label="travel_preferences", description="Diet, interests, pace.")),
    MemoryTopic(custom_memory_topic=CustomMemoryTopic(
        label="trip_context", description="Current trip: city, hotel.")),
]

## 3. Profile schema

Topics give you sentences. A schema gives you **one JSON profile per user** that your app can read by field name.

Define the fields with Pydantic, then turn it into a schema config.

In [41]:
from pydantic import BaseModel, Field


class TravelProfile(BaseModel):
    home_city: str = Field(description="Where the user lives.")
    dietary: str = Field(description="Diet, e.g. vegetarian.")
    interests: str = Field(description="Comma-separated interests.")


schema_config = {
    "id": "user-profile",
    "memory_schema": TravelProfile.model_json_schema(),
}

## 4. Create the Memory Bank

Register the topics and the schema when you create the bank. They are saved with the bank.

**Run this once.** If `AGENT_ENGINE_ID` is already set, the cell skips creation. To get a bank with these topics and this schema, clear `AGENT_ENGINE_ID` in `.env` and in the setup cell first. Creating takes a minute or two.

In [42]:
if AGENT_ENGINE_ID:
    print("Using existing engine", AGENT_ENGINE_ID)
else:
    engine = client.agent_engines.create(
        config={
            "display_name": "travel-agent-memory-2",
            "context_spec": {
                "memory_bank_config": {
                    "customization_configs": [{"memory_topics": custom_topics}],
                    "structured_memory_configs": [{"schema_configs": [schema_config]}],
                }
            },
        },
    )
    AGENT_ENGINE_ID = engine.api_resource.name.rstrip("/").split("/")[-1]
    print("Add this to .env:  AGENT_ENGINE_ID=" + AGENT_ENGINE_ID)

/Users/Bopunsiri/codespace/playgroud/memory-bank/.venv/lib/python3.14/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Add this to .env:  AGENT_ENGINE_ID=7301933960752267264


In [43]:
MEMORY_BANK = (
    AGENT_ENGINE_ID
    if AGENT_ENGINE_ID.startswith("projects/")
    else f"projects/{PROJECT_ID}/locations/{LOCATION}/reasoningEngines/{AGENT_ENGINE_ID}"
)
SCOPE = {"app_name": APP_NAME, "user_id": USER_ID}  # every memory belongs to a scope
MEMORY_BANK

'projects/data-beat-labs/locations/us-central1/reasoningEngines/7301933960752267264'

Helpers that print memories:

In [44]:
def memory_fact(item) -> str:
    memory = getattr(item, "memory", None) or item
    response = getattr(memory, "response", None)
    if response is not None:
        memory = response
    fact = getattr(memory, "fact", None)
    return fact if fact else str(item)


def memory_resource_name(item):
    memory = getattr(item, "memory", None) or item
    response = getattr(memory, "response", None)
    if response is not None:
        memory = response
    return getattr(memory, "name", None)


def show(items) -> None:
    items = list(items)
    if not items:
        print("No memories yet")
    for i, item in enumerate(items, start=1):
        print(f"{i}. {memory_fact(item)}")

## 5. Write a fact

Use this when your app already knows something, for example from a booking form. No conversation needed.

`memories.create` stores the text exactly. Running it twice gives two entries.

In [45]:
operation = client.agent_engines.memories.create(
    name=MEMORY_BANK,
    fact="Guest is traveling with a dog named Oreo.",
    scope=SCOPE,
    config={"wait_for_completion": True},
)
print(memory_fact(operation))
print(memory_resource_name(operation))

Guest is traveling with a dog named Oreo.
projects/76317526495/locations/us-central1/reasoningEngines/7301933960752267264/memories/9004984219847360512


`memories.generate` merges the fact with what's already there, so it can update an old memory instead of adding a duplicate.

In [46]:
operation = client.agent_engines.memories.generate(
    name=MEMORY_BANK,
    direct_memories_source={"direct_memories": [{"fact": "The guest enjoys exercising every morning, so she prefers a hotel near a park or with an on-site gym."}]},
    scope=SCOPE,
    config={"wait_for_completion": True},
)
operation

AgentEngineGenerateMemoriesOperation(
  done=True,
  metadata={
    '@type': 'type.googleapis.com/google.cloud.aiplatform.v1beta1.GenerateMemoriesOperationMetadata',
    'genericMetadata': {
      'createTime': '2026-11-07T03:22:37.291965Z',
      'updateTime': '2026-11-07T03:22:42.466240Z'
    }
  },
  name='projects/76317526495/locations/us-central1/reasoningEngines/7301933960752267264/operations/5071271618160885760',
  response=GenerateMemoriesResponse(
    generated_memories=[
      GenerateMemoriesResponseGeneratedMemory(
        action=<GenerateMemoriesResponseGeneratedMemoryAction.CREATED: 'CREATED'>,
        memory=Memory(
          memory_type=<MemoryType.NATURAL_LANGUAGE_COLLECTION: 'NATURAL_LANGUAGE_COLLECTION'>,
          name='projects/data-beat-labs/locations/us-central1/reasoningEngines/7301933960752267264/memories/7954519606763192320'
        )
      ),
    ]
  )
)

## 6. Read memories back

`retrieve` returns one user's memories. `list` returns every memory in the bank.

In [47]:
show(client.agent_engines.memories.retrieve(name=MEMORY_BANK, scope=SCOPE))

1. The guest enjoys exercising every morning, so she prefers a hotel near a park or with an on-site gym.
2. Guest is traveling with a dog named Oreo.


In [48]:
show(client.agent_engines.memories.list(name=MEMORY_BANK))

1. The guest enjoys exercising every morning, so she prefers a hotel near a park or with an on-site gym.
2. Guest is traveling with a dog named Oreo.


Similarity search returns the memories closest to a query.

In [49]:
show(
    client.agent_engines.memories.retrieve(
        name=MEMORY_BANK,
        scope=SCOPE,
        similarity_search_params={"search_query": "pets", "top_k": 3},
    )
)

1. Guest is traveling with a dog named Oreo.
2. The guest enjoys exercising every morning, so she prefers a hotel near a park or with an on-site gym.


## 7. Learn from a conversation

Send a chat to `generate` with `direct_contents_source`. Memory Bank sorts it into the topics and fills the profile in one call.

In [50]:
conversation = [
    {"content": {"role": "user", "parts": [{"text": "I live in Bangkok. I'm vegetarian."}]}},
    {"content": {"role": "user", "parts": [{"text": "I like walking tours."}]}},
    {"content": {"role": "user", "parts": [{"text": "Our hotel is in Shinjuku."}]}},
]

client.agent_engines.memories.generate(
    name=MEMORY_BANK,
    direct_contents_source={"events": conversation},
    scope=SCOPE,
    config={"wait_for_completion": True},
)

AgentEngineGenerateMemoriesOperation(
  done=True,
  metadata={
    '@type': 'type.googleapis.com/google.cloud.aiplatform.v1beta1.GenerateMemoriesOperationMetadata',
    'genericMetadata': {
      'createTime': '2026-11-07T03:23:00.628385Z',
      'updateTime': '2026-11-07T03:23:17.707204Z'
    }
  },
  name='projects/76317526495/locations/us-central1/reasoningEngines/7301933960752267264/operations/743312376257839104',
  response=GenerateMemoriesResponse(
    generated_memories=[
      GenerateMemoriesResponseGeneratedMemory(
        action=<GenerateMemoriesResponseGeneratedMemoryAction.CREATED: 'CREATED'>,
        memory=Memory(
          memory_type=<MemoryType.NATURAL_LANGUAGE_COLLECTION: 'NATURAL_LANGUAGE_COLLECTION'>,
          name='projects/data-beat-labs/locations/us-central1/reasoningEngines/7301933960752267264/memories/6954720489486942208'
        )
      ),
      GenerateMemoriesResponseGeneratedMemory(
        action=<GenerateMemoriesResponseGeneratedMemoryAction.CREATED: '

Sentences from the topics:

In [36]:
show(client.agent_engines.memories.retrieve(name=MEMORY_BANK, scope=SCOPE))

1. My hotel is located in Shinjuku.
2. I am a vegetarian.
3. Guest is traveling with a dog named Oreo.
4. The guest enjoys exercising every morning, so she prefers a hotel near a park or with an on-site gym.
5. Guest is traveling with a dog named Oreo.


The profile from the schema. Expect `home_city: Bangkok`, `dietary: vegetarian`, `interests: walking tours`. If it's empty, wait a moment and run again.

In [51]:
result = client.agent_engines.memories.retrieve_profiles(name=MEMORY_BANK, scope=SCOPE)
for schema_id, profile in (result.profiles or {}).items():
    print(schema_id, profile.profile)

user-profile {'home_city': 'Bangkok', 'dietary': 'vegetarian', 'interests': 'walking tours'}


## 8. Clean up

Both cells are commented out so they don't run by accident.

In [ ]:
for item in client.agent_engines.memories.retrieve(name=MEMORY_BANK, scope=SCOPE):
    client.agent_engines.memories.delete(name=memory_resource_name(item))
show(client.agent_engines.memories.retrieve(name=MEMORY_BANK, scope=SCOPE))

In [ ]:
client.agent_engines.delete(name=MEMORY_BANK, force=True)